# S07 · Inspect blank nodes and OWL lists

**Outcome:** Read the RDF serialization of restrictions and lists without confusing that structure with its logical meaning.

**Time:** about 40 minutes. Run cells in order. This is the executed solution edition.

OWL class expressions are often serialized using blank nodes and RDF collections. A blank-node identifier is local to a parse or graph and is not a durable business key. Use structural patterns, graph isomorphism or stable source identifiers for comparison rather than persisting an arbitrary blank-node label.

The list path rdf:rest*/rdf:first retrieves members of a serialized collection. For owl:unionOf those members describe alternatives; the RDF list structure alone does not classify individuals. An anonymous class is still queryable as an RDF node using variables and structural patterns. Giving the expression a named equivalent class can make consumer queries easier, but lack of a name does not make it impossible to query.

This lesson corrects an overstatement in the pwin inference commentary: anonymous union classes can be inspected and matched. It also retains the useful lesson that the domain of a property is an inference rule, not an input validation constraint. Keep list inspection, RDFS entailment and OWL DL interpretation conceptually separate.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Traverse the union expression as RDF

In [2]:
g=schema()
query_text='''PREFIX owl: <http://www.w3.org/2002/07/owl#>
SELECT ?member WHERE {
 ex:CardiopulmonaryRecord owl:equivalentClass ?expr .
 ?expr owl:unionOf/rdf:rest*/rdf:first ?member .
} ORDER BY ?member'''
union_members={str(row[0]) for row in query(g,query_text)}
assert union_members=={str(EX.CardiacCodedRecord),str(EX.RespiratoryCodedRecord)}
display(union_members)

{'https://example.org/health/ontology/CardiacCodedRecord',
 'https://example.org/health/ontology/RespiratoryCodedRecord'}

## Round-trip without relying on blank-node labels

In [3]:
from rdflib.compare import isomorphic
roundtrip=Graph().parse(data=g.serialize(format='turtle'),format='turtle')
assert isomorphic(g,roundtrip)
print('Graph structure is preserved; blank-node spelling need not be.')

Graph structure is preserved; blank-node spelling need not be.


## Your turn

Return the two property IRIs that occur in the aboutPatient property chain.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = [row[0] for row in query(schema(),'PREFIX owl: <http://www.w3.org/2002/07/owl#> SELECT ?p WHERE { ex:aboutPatient owl:propertyChainAxiom/rdf:rest*/rdf:first ?p }')]

In [5]:
learner_check(answer, lambda x:set(x)=={EX.documents,EX.hasPatient}, 'This query retrieves members; use explicit rdf:first/rest positions if order matters.')

Exercise passed.
Out[0]: True


## Explain your model

Why is a property chain ordered even though the set returned by a path query is not an ordered list?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** Compare the proof premises, source scope and query contract described above; use your own words in a peer review.